In [1]:
import pickle, numpy as np
import matplotlib.pyplot as plt

# ========== 1) 读取并解析结构 ==========
P_PATH = "samples_small.pkl"  # TODO: 改成你的实际路径

with open(P_PATH, "rb") as f:
    raw = pickle.load(f)

def _to_arm_samples(raw_obj):
    """
    尝试把各种常见结构解析成：
      arm_samples: dict[int -> np.ndarray]  形如 {0: [x...], 1: [x...], ...}
    若有多日期：合并所有日期（也返回 per-date 以便检查非平稳）
    """
    arm_samples = {}
    per_date = []  # list of dict per date
    # 情况 A: {date: {arm: np.array}}
    if isinstance(raw_obj, dict):
        # 可能 keys 是日期或臂
        sample_like = lambda v: isinstance(v, (list, tuple, np.ndarray))
        # 检测二层 dict
        if raw_obj and all(isinstance(v, dict) for v in raw_obj.values()):
            # 视为 {date: {arm: samples}}
            for dkey, dd in raw_obj.items():
                dd_clean = {}
                for akey, arr in dd.items():
                    try:
                        aid = int(akey)
                    except:
                        # 若键是 'arm_0' 这类，尝试解析末尾数字
                        s = str(akey)
                        digits = "".join(ch for ch in s if ch.isdigit())
                        aid = int(digits) if digits else s
                    arr = np.asarray(arr, dtype=float).ravel()
                    dd_clean[aid] = arr
                    arm_samples.setdefault(aid, [])
                    arm_samples[aid].append(arr)
                per_date.append(dd_clean)
            # 合并到单一 dict
            for aid, arr_list in arm_samples.items():
                arm_samples[aid] = np.concatenate(arr_list) if arr_list else np.array([], dtype=float)
            return arm_samples, per_date
        else:
            # 视为 {arm: samples}
            for akey, arr in raw_obj.items():
                try:
                    aid = int(akey)
                except:
                    s = str(akey)
                    digits = "".join(ch for ch in s if ch.isdigit())
                    aid = int(digits) if digits else s
                arm_samples[aid] = np.asarray(arr, dtype=float).ravel()
            return arm_samples, []
    # 情况 B: list/tuple of per-arm arrays
    if isinstance(raw_obj, (list, tuple)) and raw_obj and all(isinstance(x, (list, tuple, np.ndarray)) for x in raw_obj):
        for i, arr in enumerate(raw_obj):
            arm_samples[i] = np.asarray(arr, dtype=float).ravel()
        return arm_samples, []
    raise ValueError("Unrecognized samples.pkl structure; please print(type/raw) to inspect.")

arm_samples, per_date = _to_arm_samples(raw)
K = len(arm_samples)
print(f"Parsed arms: K={K}. Per-date blocks: {len(per_date)}")

# ========== 2) 基本统计 ==========
def _stats(x):
    x = np.asarray(x, dtype=float).ravel()
    n = x.size
    mu = float(np.mean(x)) if n else np.nan
    var = float(np.var(x)) if n else np.nan
    std = np.sqrt(var) if n else np.nan
    # 偏度/峰度（无偏校正可以忽略，在线调参够用）
    m = mu
    s3 = np.mean((x - m)**3) if n else np.nan
    s4 = np.mean((x - m)**4) if n else np.nan
    skew = s3 / (std**3 + 1e-12) if n else np.nan
    kurt = s4 / (var**2 + 1e-12) if n else np.nan  # 总体峰度（Fisher/过峰= kurt-3）
    q = np.quantile(x, [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]) if n else [np.nan]*7
    out_lo = float(np.mean(x < 0.0)) if n else np.nan
    out_hi = float(np.mean(x > 1.0)) if n else np.nan
    return dict(n=n, mean=mu, var=var, std=std, skew=skew, kurt=kurt,
                q01=q[0], q05=q[1], q25=q[2], q50=q[3], q75=q[4], q95=q[5], q99=q[6],
                frac_below0=out_lo, frac_above1=out_hi, min=float(np.min(x)) if n else np.nan, max=float(np.max(x)) if n else np.nan)

summary = {aid: _stats(arr) for aid, arr in arm_samples.items()}
# 打印成表
import math
print("\nPer-arm summary:")
print("arm  n       mean     std      var      skew     kurt     q05      q50      q95     min     max   out<0  out>1")
for aid in sorted(summary):
    s = summary[aid]
    print(f"{aid:>3}  {s['n']:>7d}  {s['mean']:>7.4f}  {s['std']:>7.4f}  {s['var']:>7.4f}  {s['skew']:>7.3f}  {s['kurt']:>7.3f}  "
          f"{s['q05']:>7.4f}  {s['q50']:>7.4f}  {s['q95']:>7.4f}  {s['min']:>6.3f}  {s['max']:>6.3f}  {s['frac_below0']:>6.3f}  {s['frac_above1']:>6.3f}")

# ========== 3) 估算最优臂与 gaps + 自助法置信区间 ==========
rng = np.random.RandomState(123)
def bootstrap_mean_ci(x, B=2000, alpha=0.05):
    x = np.asarray(x, float).ravel()
    n = x.size
    if n == 0: return (np.nan, np.nan, np.nan)
    idx = rng.randint(0, n, size=(B, n))
    samples = x[idx]
    means = samples.mean(axis=1)
    lo, hi = np.quantile(means, [alpha/2, 1 - alpha/2])
    return float(np.mean(x)), float(lo), float(hi)

means = {aid: np.mean(arr) for aid, arr in arm_samples.items()}
best_arm = max(means, key=means.get)
best_mean = means[best_arm]
gaps = {aid: float(best_mean - m) for aid, m in means.items()}

print(f"\nEstimated best arm: {best_arm} with mean {best_mean:.4f}")
print("Gaps (best_mean - mean_i):")
for aid in sorted(gaps): print(f"  arm {aid}: gap={gaps[aid]:.4f}")

print("\nBootstrap 95% CIs for means:")
for aid in sorted(arm_samples):
    mu, lo, hi = bootstrap_mean_ci(arm_samples[aid], B=1000)
    print(f"  arm {aid}: mean={mu:.4f}, 95% CI=({lo:.4f}, {hi:.4f})")

# ========== 4) 可视化（ECDF & 直方图） ==========
def ecdf(x):
    x = np.sort(np.asarray(x, float).ravel())
    n = x.size
    y = np.arange(1, n+1) / n
    return x, y

# ECDF
plt.figure()
for aid in sorted(arm_samples):
    xs, ys = ecdf(arm_samples[aid])
    plt.step(xs, ys, where="post", label=f"arm {aid}")
plt.title("Per-arm ECDF")
plt.xlabel("reward")
plt.ylabel("ECDF")
plt.legend()
plt.show()

# Hist
for aid in sorted(arm_samples):
    plt.figure()
    plt.hist(arm_samples[aid], bins=30, density=True)
    plt.title(f"Arm {aid} histogram")
    plt.xlabel("reward")
    plt.ylabel("density")
    plt.show()

# ========== 5)（可选）跨日期非平稳性检查 ==========
if per_date:
    print(f"\nNon-stationarity check across {len(per_date)} dates (per-arm means):")
    per_date_means = []
    for d, dd in enumerate(per_date):
        line = {}
        for aid in sorted(arm_samples):
            if aid in dd and dd[aid].size > 0:
                line[aid] = float(np.mean(dd[aid]))
            else:
                line[aid] = np.nan
        per_date_means.append(line)
        print(f"  date {d}: " + " ".join([f"arm{aid}:{line[aid]:.4f}" if not np.isnan(line[aid]) else f"arm{aid}:nan" for aid in sorted(line)]))


ValueError: Unrecognized samples.pkl structure; please print(type/raw) to inspect.